# CUDA Chess Engine Colab Benchmark

This notebook rebuilds the project from embedded source, verifies move generation, and benchmarks serial, OpenMP, and CUDA search paths.

Before running the build cell, switch Colab to a GPU runtime.


## 1. Detect GPU Runtime


In [ ]:
import subprocess

query_cmd = [
    'nvidia-smi',
    '--query-gpu=name,memory.total,driver_version,compute_cap',
    '--format=csv,noheader',
]
result = subprocess.run(query_cmd, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No NVIDIA GPU runtime detected. In Colab, switch to Runtime > Change runtime type > GPU.'
    )

gpu_rows = [line.strip() for line in result.stdout.splitlines() if line.strip()]
for idx, row in enumerate(gpu_rows, start=1):
    print(f'GPU {idx}: {row}')

first_gpu_cols = [part.strip() for part in gpu_rows[0].split(',')]
compute_cap = first_gpu_cols[-1] if first_gpu_cols else '7.5'

digits = ''.join(ch for ch in compute_cap if ch.isdigit())
cuda_arch = digits or '75'
print(f'Using CMAKE_CUDA_ARCHITECTURES={cuda_arch}')


## 2. Write Project Files

The notebook writes all `14` tracked source files into `/content/chess`.


In [ ]:
from pathlib import Path

project_dir = Path('/content/chess')
project_dir.mkdir(parents=True, exist_ok=True)
(project_dir / 'src').mkdir(exist_ok=True)
(project_dir / 'cuda').mkdir(exist_ok=True)

files = {}

files['src/board.h'] = '#pragma once\n#include <cstdint>\n#include <string>\n\n// Square layout: A1=0, B1=1, ..., H1=7, A2=8, ..., H8=63\n//   rank 8 → squares 56-63 (top of board)\n//   rank 1 → squares  0- 7 (bottom of board)\ninline constexpr int rank_of(int s) { return s >> 3; }\ninline constexpr int file_of(int s) { return s & 7; }\ninline constexpr int make_sq(int r, int f) { return (r << 3) | f; }\n\nenum : int {\n    A1=0, B1, C1, D1, E1, F1, G1, H1,\n    A2=8, B2, C2, D2, E2, F2, G2, H2,\n    A3=16,B3, C3, D3, E3, F3, G3, H3,\n    A4=24,B4, C4, D4, E4, F4, G4, H4,\n    A5=32,B5, C5, D5, E5, F5, G5, H5,\n    A6=40,B6, C6, D6, E6, F6, G6, H6,\n    A7=48,B7, C7, D7, E7, F7, G7, H7,\n    A8=56,B8, C8, D8, E8, F8, G8, H8,\n};\n\n// Piece encoding: positive=white, negative=black, 0=empty\n// |piece|: 1=pawn 2=knight 3=bishop 4=rook 5=queen 6=king\nenum Piece : int8_t {\n    B_KING=-6, B_QUEEN=-5, B_ROOK=-4, B_BISHOP=-3, B_KNIGHT=-2, B_PAWN=-1,\n    EMPTY=0,\n    W_PAWN=1, W_KNIGHT=2, W_BISHOP=3, W_ROOK=4, W_QUEEN=5, W_KING=6\n};\n\ninline int8_t piece_type(int8_t p) { return p < 0 ? -p : p; }\n\n// Castling rights bit flags\nconstexpr uint8_t CR_WK = 1, CR_WQ = 2, CR_BK = 4, CR_BQ = 8;\n\nstruct Move {\n    uint8_t from;\n    uint8_t to;\n    int8_t  promo;  // 0 or piece type (always positive: KNIGHT/BISHOP/ROOK/QUEEN)\n    uint8_t flags;\n\n    enum Flags : uint8_t {\n        CAPTURE  = 0x01,\n        EN_PASS  = 0x02,\n        CASTLE   = 0x04,\n        DBL_PUSH = 0x08,\n    };\n\n    bool is_null() const { return from == 0 && to == 0 && promo == 0 && flags == 0; }\n    static Move null() { return {0, 0, 0, 0}; }\n};\n\nstruct UndoInfo {\n    int8_t  captured;\n    uint8_t castle;\n    int8_t  ep;\n    int     half;\n};\n\nstruct Board {\n    int8_t  sq[64]{};\n    bool    wtm{true};   // white to move\n    uint8_t castle{0};   // castling rights\n    int8_t  ep{-1};      // en passant target square, -1 = none\n    int     half{0};     // halfmove clock (50-move rule)\n    int     full{1};     // fullmove number\n\n    void        reset();\n    void        set_startpos();\n    bool        load_fen(const std::string& fen);\n    std::string to_fen() const;\n    void        print() const;\n\n    UndoInfo make_move(const Move& m);\n    void     undo_move(const Move& m, const UndoInfo& u);\n\n    int  king_sq(bool white) const;\n    bool square_attacked(int s, bool by_white) const;\n    bool in_check() const;\n};\n'

files['src/board.cpp'] = '#include "board.h"\n#include <cctype>\n#include <cstdio>\n#include <cstring>\n#include <sstream>\n#include <algorithm>\n\nvoid Board::reset() {\n    std::fill(sq, sq + 64, (int8_t)EMPTY);\n    wtm    = true;\n    castle = 0;\n    ep     = -1;\n    half   = 0;\n    full   = 1;\n}\n\nvoid Board::set_startpos() {\n    load_fen("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1");\n}\n\nbool Board::load_fen(const std::string& fen) {\n    reset();\n    std::istringstream ss(fen);\n    std::string pieces, side, castle_str, ep_str;\n    int hm = 0, fm = 1;\n\n    if (!(ss >> pieces >> side >> castle_str >> ep_str >> hm >> fm))\n        return false;\n\n    // Piece placement: FEN starts from rank 8 (r=7) down to rank 1 (r=0)\n    int r = 7, f = 0;\n    for (char c : pieces) {\n        if (c == \'/\') { r--; f = 0; }\n        else if (std::isdigit(c)) { f += c - \'0\'; }\n        else {\n            int8_t p = EMPTY;\n            switch (std::toupper(c)) {\n                case \'P\': p = W_PAWN;   break;\n                case \'N\': p = W_KNIGHT; break;\n                case \'B\': p = W_BISHOP; break;\n                case \'R\': p = W_ROOK;   break;\n                case \'Q\': p = W_QUEEN;  break;\n                case \'K\': p = W_KING;   break;\n            }\n            if (std::islower(c)) p = -p;\n            sq[make_sq(r, f++)] = p;\n        }\n    }\n\n    wtm = (side == "w");\n\n    castle = 0;\n    for (char c : castle_str) {\n        switch (c) {\n            case \'K\': castle |= CR_WK; break;\n            case \'Q\': castle |= CR_WQ; break;\n            case \'k\': castle |= CR_BK; break;\n            case \'q\': castle |= CR_BQ; break;\n        }\n    }\n\n    ep = -1;\n    if (ep_str != "-")\n        ep = make_sq(ep_str[1] - \'1\', ep_str[0] - \'a\');\n\n    half = hm;\n    full = fm;\n    return true;\n}\n\nstd::string Board::to_fen() const {\n    std::string fen;\n    for (int r = 7; r >= 0; r--) {\n        int empty = 0;\n        for (int f = 0; f < 8; f++) {\n            int8_t p = sq[make_sq(r, f)];\n            if (p == EMPTY) { empty++; continue; }\n            if (empty) { fen += char(\'0\' + empty); empty = 0; }\n            int8_t t = piece_type(p);\n            char c = "PNBRQK"[t - 1];\n            fen += (p > 0) ? c : char(std::tolower(c));\n        }\n        if (empty) fen += char(\'0\' + empty);\n        if (r > 0)  fen += \'/\';\n    }\n    fen += wtm ? " w " : " b ";\n    std::string cr;\n    if (castle & CR_WK) cr += \'K\';\n    if (castle & CR_WQ) cr += \'Q\';\n    if (castle & CR_BK) cr += \'k\';\n    if (castle & CR_BQ) cr += \'q\';\n    fen += cr.empty() ? "-" : cr;\n    fen += \' \';\n    if (ep < 0) fen += \'-\';\n    else { fen += char(\'a\' + file_of(ep)); fen += char(\'1\' + rank_of(ep)); }\n    fen += \' \'; fen += std::to_string(half);\n    fen += \' \'; fen += std::to_string(full);\n    return fen;\n}\n\nvoid Board::print() const {\n    printf("\\n");\n    for (int r = 7; r >= 0; r--) {\n        printf(" %d | ", r + 1);\n        for (int f = 0; f < 8; f++) {\n            int8_t p = sq[make_sq(r, f)];\n            char c;\n            if (p == 0) c = \'.\';\n            else if (p > 0) c = "PNBRQK"[p - 1];\n            else             c = "pnbrqk"[-p - 1];\n            printf("%c ", c);\n        }\n        printf("\\n");\n    }\n    printf("     ----------------\\n");\n    printf("     a b c d e f g h\\n\\n");\n    char ep_str[3] = "-";\n    if (ep >= 0) { ep_str[0] = \'a\' + file_of(ep); ep_str[1] = \'1\' + rank_of(ep); ep_str[2] = \'\\0\'; }\n    printf("%s to move | castle: %c%c%c%c | ep: %s | half: %d | full: %d\\n",\n           wtm ? "White" : "Black",\n           (castle & CR_WK) ? \'K\' : \'-\',\n           (castle & CR_WQ) ? \'Q\' : \'-\',\n           (castle & CR_BK) ? \'k\' : \'-\',\n           (castle & CR_BQ) ? \'q\' : \'-\',\n           ep_str, half, full);\n}\n\n// Revoke castling rights when pieces move to/from key squares\nstatic void revoke_castle(uint8_t& castle, int sq_idx) {\n    switch (sq_idx) {\n        case E1: castle &= ~(CR_WK | CR_WQ); break;\n        case E8: castle &= ~(CR_BK | CR_BQ); break;\n        case A1: castle &= ~CR_WQ; break;\n        case H1: castle &= ~CR_WK; break;\n        case A8: castle &= ~CR_BQ; break;\n        case H8: castle &= ~CR_BK; break;\n        default: break;\n    }\n}\n\nUndoInfo Board::make_move(const Move& m) {\n    UndoInfo u{sq[m.to], castle, ep, half};\n    int8_t moving = sq[m.from];\n\n    ep = -1;\n\n    // Halfmove clock: reset on pawn move or capture\n    if (piece_type(moving) == 1 || (m.flags & Move::CAPTURE))\n        half = 0;\n    else\n        half++;\n\n    // En passant capture: captured pawn is not on m.to\n    if (m.flags & Move::EN_PASS) {\n        int cap_sq = m.to + (wtm ? -8 : 8);\n        u.captured = sq[cap_sq];\n        sq[cap_sq] = EMPTY;\n    }\n\n    // Move piece (handle promotion)\n    if (m.promo)\n        sq[m.to] = wtm ? m.promo : -m.promo;\n    else\n        sq[m.to] = moving;\n    sq[m.from] = EMPTY;\n\n    // Move rook when castling\n    if (m.flags & Move::CASTLE) {\n        switch (m.to) {\n            case G1: sq[F1] = W_ROOK; sq[H1] = EMPTY; break;\n            case C1: sq[D1] = W_ROOK; sq[A1] = EMPTY; break;\n            case G8: sq[F8] = B_ROOK; sq[H8] = EMPTY; break;\n            case C8: sq[D8] = B_ROOK; sq[A8] = EMPTY; break;\n        }\n    }\n\n    // Set en passant square for double pawn push\n    if (m.flags & Move::DBL_PUSH)\n        ep = m.to + (wtm ? -8 : 8);\n\n    // Update castling rights based on squares touched\n    revoke_castle(castle, m.from);\n    revoke_castle(castle, m.to);\n\n    if (!wtm) full++;\n    wtm = !wtm;\n    return u;\n}\n\nvoid Board::undo_move(const Move& m, const UndoInfo& u) {\n    wtm = !wtm;\n    if (!wtm) full--;\n\n    // The piece on m.to might be a promoted piece — restore original pawn\n    int8_t moving = m.promo ? (wtm ? W_PAWN : B_PAWN) : sq[m.to];\n\n    sq[m.from] = moving;\n\n    // Restore destination (may be EMPTY for EP, otherwise captured piece)\n    sq[m.to] = (m.flags & Move::EN_PASS) ? EMPTY : u.captured;\n\n    // Restore en passant captured pawn\n    if (m.flags & Move::EN_PASS) {\n        int cap_sq = m.to + (wtm ? -8 : 8);\n        sq[cap_sq] = u.captured;\n    }\n\n    // Undo castling rook move\n    if (m.flags & Move::CASTLE) {\n        switch (m.to) {\n            case G1: sq[H1] = W_ROOK; sq[F1] = EMPTY; break;\n            case C1: sq[A1] = W_ROOK; sq[D1] = EMPTY; break;\n            case G8: sq[H8] = B_ROOK; sq[F8] = EMPTY; break;\n            case C8: sq[A8] = B_ROOK; sq[D8] = EMPTY; break;\n        }\n    }\n\n    castle = u.castle;\n    ep     = u.ep;\n    half   = u.half;\n}\n\nint Board::king_sq(bool white) const {\n    int8_t king = white ? W_KING : B_KING;\n    for (int s = 0; s < 64; s++)\n        if (sq[s] == king) return s;\n    return -1;\n}\n\nbool Board::square_attacked(int s, bool by_white) const {\n    const int8_t sign = by_white ? 1 : -1;\n    const int r = rank_of(s), f = file_of(s);\n\n    // Pawn attacks: check if a pawn of the attacking side attacks s\n    // A white pawn on (r-1, f±1) attacks s\n    {\n        int pr = by_white ? r - 1 : r + 1;\n        if (pr >= 0 && pr < 8) {\n            for (int df : {-1, 1}) {\n                int pf = f + df;\n                if (pf >= 0 && pf < 8 && sq[make_sq(pr, pf)] == sign * W_PAWN)\n                    return true;\n            }\n        }\n    }\n\n    // Knight attacks\n    static const int8_t KNR[8] = {-2,-2,-1,-1, 1, 1, 2, 2};\n    static const int8_t KNF[8] = {-1, 1,-2, 2,-2, 2,-1, 1};\n    for (int i = 0; i < 8; i++) {\n        int nr = r + KNR[i], nf = f + KNF[i];\n        if (nr >= 0 && nr < 8 && nf >= 0 && nf < 8)\n            if (sq[make_sq(nr, nf)] == sign * W_KNIGHT) return true;\n    }\n\n    // Diagonal sliders (bishop / queen)\n    static const int8_t DR[4] = { 1, 1,-1,-1};\n    static const int8_t DF[4] = { 1,-1, 1,-1};\n    for (int d = 0; d < 4; d++) {\n        int cr = r + DR[d], cf = f + DF[d];\n        while (cr >= 0 && cr < 8 && cf >= 0 && cf < 8) {\n            int8_t p = sq[make_sq(cr, cf)];\n            if (p != EMPTY) {\n                if (p == sign * W_BISHOP || p == sign * W_QUEEN) return true;\n                break;\n            }\n            cr += DR[d]; cf += DF[d];\n        }\n    }\n\n    // Straight sliders (rook / queen)\n    static const int8_t SR[4] = { 1,-1, 0, 0};\n    static const int8_t SF[4] = { 0, 0, 1,-1};\n    for (int d = 0; d < 4; d++) {\n        int cr = r + SR[d], cf = f + SF[d];\n        while (cr >= 0 && cr < 8 && cf >= 0 && cf < 8) {\n            int8_t p = sq[make_sq(cr, cf)];\n            if (p != EMPTY) {\n                if (p == sign * W_ROOK || p == sign * W_QUEEN) return true;\n                break;\n            }\n            cr += SR[d]; cf += SF[d];\n        }\n    }\n\n    // King attacks\n    for (int dr = -1; dr <= 1; dr++)\n    for (int df = -1; df <= 1; df++) {\n        if (!dr && !df) continue;\n        int nr = r + dr, nf = f + df;\n        if (nr >= 0 && nr < 8 && nf >= 0 && nf < 8)\n            if (sq[make_sq(nr, nf)] == sign * W_KING) return true;\n    }\n\n    return false;\n}\n\nbool Board::in_check() const {\n    return square_attacked(king_sq(wtm), !wtm);\n}\n'

files['src/movegen.h'] = '#pragma once\n#include "board.h"\n#include <vector>\n\n// Generate all pseudo-legal moves (may leave own king in check)\nstd::vector<Move> generate_pseudo_legal(const Board& b);\n\n// Generate all legal moves (filters out moves that leave own king in check)\nstd::vector<Move> generate_legal_moves(Board& b);\n\n// Format a move as UCI string, e.g. "e2e4", "e7e8q"\nstd::string move_to_str(const Move& m);\n'

files['src/movegen.cpp'] = '#include "movegen.h"\n#include <vector>\n\nusing ML = std::vector<Move>;\n\n// ── Pawns ─────────────────────────────────────────────────────────────────\n\nstatic void push_promos(ML& ml, uint8_t from, uint8_t to, uint8_t flags) {\n    for (int8_t p : {W_QUEEN, W_ROOK, W_BISHOP, W_KNIGHT})\n        ml.push_back({from, to, p, flags});\n}\n\nstatic void gen_pawns(const Board& b, ML& ml) {\n    const int    dir    = b.wtm ?  8 : -8;\n    const int    start  = b.wtm ?  1 :  6;  // starting rank index\n    const int    prank  = b.wtm ?  6 :  1;  // rank before promotion\n    const int8_t pawn   = b.wtm ? W_PAWN : B_PAWN;\n\n    for (int s = 0; s < 64; s++) {\n        if (b.sq[s] != pawn) continue;\n        const int r = rank_of(s), f = file_of(s);\n        const int fwd = s + dir;\n\n        // Forward push\n        if (b.sq[fwd] == EMPTY) {\n            if (r == prank)\n                push_promos(ml, s, fwd, 0);\n            else\n                ml.push_back({(uint8_t)s, (uint8_t)fwd, 0, 0});\n\n            // Double push from starting rank\n            if (r == start && b.sq[fwd + dir] == EMPTY)\n                ml.push_back({(uint8_t)s, (uint8_t)(fwd + dir), 0, Move::DBL_PUSH});\n        }\n\n        // Diagonal captures and en passant\n        for (int df : {-1, 1}) {\n            int nf = f + df;\n            if (nf < 0 || nf >= 8) continue;\n            int cap = fwd + df;\n\n            if (cap == (int)b.ep) {\n                ml.push_back({(uint8_t)s, (uint8_t)cap, 0, Move::EN_PASS | Move::CAPTURE});\n            } else if (b.sq[cap] != EMPTY && ((b.sq[cap] > 0) != b.wtm)) {\n                // Enemy piece\n                if (r == prank)\n                    push_promos(ml, s, cap, Move::CAPTURE);\n                else\n                    ml.push_back({(uint8_t)s, (uint8_t)cap, 0, Move::CAPTURE});\n            }\n        }\n    }\n}\n\n// ── Knights ───────────────────────────────────────────────────────────────\n\nstatic void gen_knights(const Board& b, ML& ml) {\n    static const int8_t DR[8] = {-2,-2,-1,-1, 1, 1, 2, 2};\n    static const int8_t DF[8] = {-1, 1,-2, 2,-2, 2,-1, 1};\n    const int8_t knight = b.wtm ? W_KNIGHT : B_KNIGHT;\n\n    for (int s = 0; s < 64; s++) {\n        if (b.sq[s] != knight) continue;\n        int r = rank_of(s), f = file_of(s);\n        for (int i = 0; i < 8; i++) {\n            int nr = r + DR[i], nf = f + DF[i];\n            if (nr < 0 || nr >= 8 || nf < 0 || nf >= 8) continue;\n            int ns = make_sq(nr, nf);\n            if (b.sq[ns] != EMPTY && ((b.sq[ns] > 0) == b.wtm)) continue; // friendly\n            uint8_t flags = (b.sq[ns] != EMPTY) ? Move::CAPTURE : 0;\n            ml.push_back({(uint8_t)s, (uint8_t)ns, 0, flags});\n        }\n    }\n}\n\n// ── Sliding pieces ────────────────────────────────────────────────────────\n\nstatic void slide(const Board& b, ML& ml, int s, const int8_t* dr, const int8_t* df, int ndirs) {\n    int r = rank_of(s), f = file_of(s);\n    for (int d = 0; d < ndirs; d++) {\n        int cr = r + dr[d], cf = f + df[d];\n        while (cr >= 0 && cr < 8 && cf >= 0 && cf < 8) {\n            int ns = make_sq(cr, cf);\n            if (b.sq[ns] != EMPTY) {\n                if ((b.sq[ns] > 0) != b.wtm) // enemy\n                    ml.push_back({(uint8_t)s, (uint8_t)ns, 0, Move::CAPTURE});\n                break;\n            }\n            ml.push_back({(uint8_t)s, (uint8_t)ns, 0, 0});\n            cr += dr[d]; cf += df[d];\n        }\n    }\n}\n\nstatic void gen_bishops(const Board& b, ML& ml) {\n    static const int8_t DR[4] = { 1, 1,-1,-1};\n    static const int8_t DF[4] = { 1,-1, 1,-1};\n    const int8_t bishop = b.wtm ? W_BISHOP : B_BISHOP;\n    for (int s = 0; s < 64; s++)\n        if (b.sq[s] == bishop) slide(b, ml, s, DR, DF, 4);\n}\n\nstatic void gen_rooks(const Board& b, ML& ml) {\n    static const int8_t DR[4] = { 1,-1, 0, 0};\n    static const int8_t DF[4] = { 0, 0, 1,-1};\n    const int8_t rook = b.wtm ? W_ROOK : B_ROOK;\n    for (int s = 0; s < 64; s++)\n        if (b.sq[s] == rook) slide(b, ml, s, DR, DF, 4);\n}\n\nstatic void gen_queens(const Board& b, ML& ml) {\n    static const int8_t DR[8] = { 1,-1, 0, 0, 1, 1,-1,-1};\n    static const int8_t DF[8] = { 0, 0, 1,-1, 1,-1, 1,-1};\n    const int8_t queen = b.wtm ? W_QUEEN : B_QUEEN;\n    for (int s = 0; s < 64; s++)\n        if (b.sq[s] == queen) slide(b, ml, s, DR, DF, 8);\n}\n\n// ── King ──────────────────────────────────────────────────────────────────\n\nstatic void gen_king(const Board& b, ML& ml) {\n    const int8_t king = b.wtm ? W_KING : B_KING;\n\n    for (int s = 0; s < 64; s++) {\n        if (b.sq[s] != king) continue;\n        int r = rank_of(s), f = file_of(s);\n\n        // Normal king moves\n        for (int dr = -1; dr <= 1; dr++)\n        for (int df = -1; df <= 1; df++) {\n            if (!dr && !df) continue;\n            int nr = r + dr, nf = f + df;\n            if (nr < 0 || nr >= 8 || nf < 0 || nf >= 8) continue;\n            int ns = make_sq(nr, nf);\n            if (b.sq[ns] != EMPTY && ((b.sq[ns] > 0) == b.wtm)) continue; // friendly\n            uint8_t flags = (b.sq[ns] != EMPTY) ? Move::CAPTURE : 0;\n            ml.push_back({(uint8_t)s, (uint8_t)ns, 0, flags});\n        }\n\n        // Castling (also checks that king doesn\'t pass through check)\n        bool opp = !b.wtm;\n        if (b.wtm) {\n            if ((b.castle & CR_WK) && b.sq[F1] == EMPTY && b.sq[G1] == EMPTY\n                && !b.square_attacked(E1, opp)\n                && !b.square_attacked(F1, opp)\n                && !b.square_attacked(G1, opp))\n                ml.push_back({E1, G1, 0, Move::CASTLE});\n\n            if ((b.castle & CR_WQ) && b.sq[D1] == EMPTY && b.sq[C1] == EMPTY && b.sq[B1] == EMPTY\n                && !b.square_attacked(E1, opp)\n                && !b.square_attacked(D1, opp)\n                && !b.square_attacked(C1, opp))\n                ml.push_back({E1, C1, 0, Move::CASTLE});\n        } else {\n            if ((b.castle & CR_BK) && b.sq[F8] == EMPTY && b.sq[G8] == EMPTY\n                && !b.square_attacked(E8, opp)\n                && !b.square_attacked(F8, opp)\n                && !b.square_attacked(G8, opp))\n                ml.push_back({E8, G8, 0, Move::CASTLE});\n\n            if ((b.castle & CR_BQ) && b.sq[D8] == EMPTY && b.sq[C8] == EMPTY && b.sq[B8] == EMPTY\n                && !b.square_attacked(E8, opp)\n                && !b.square_attacked(D8, opp)\n                && !b.square_attacked(C8, opp))\n                ml.push_back({E8, C8, 0, Move::CASTLE});\n        }\n        break; // only one king\n    }\n}\n\n// ── Public API ────────────────────────────────────────────────────────────\n\nstd::vector<Move> generate_pseudo_legal(const Board& b) {\n    ML ml;\n    ml.reserve(48);\n    gen_pawns(b, ml);\n    gen_knights(b, ml);\n    gen_bishops(b, ml);\n    gen_rooks(b, ml);\n    gen_queens(b, ml);\n    gen_king(b, ml);\n    return ml;\n}\n\nstd::vector<Move> generate_legal_moves(Board& b) {\n    auto pseudo = generate_pseudo_legal(b);\n    ML legal;\n    legal.reserve(pseudo.size());\n    for (const auto& m : pseudo) {\n        auto u = b.make_move(m);\n        // After make_move, b.wtm is flipped; check king of the side that just moved\n        if (!b.square_attacked(b.king_sq(!b.wtm), b.wtm))\n            legal.push_back(m);\n        b.undo_move(m, u);\n    }\n    return legal;\n}\n\nstd::string move_to_str(const Move& m) {\n    std::string s;\n    s += char(\'a\' + file_of(m.from));\n    s += char(\'1\' + rank_of(m.from));\n    s += char(\'a\' + file_of(m.to));\n    s += char(\'1\' + rank_of(m.to));\n    if (m.promo) {\n        const char* pchars = "  nbrq";\n        s += pchars[m.promo]; // promo is 2=N,3=B,4=R,5=Q\n    }\n    return s;\n}\n'

files['src/eval.h'] = '#pragma once\n#include "board.h"\n\n// Static evaluation from white\'s perspective (centipawns).\n// Positive = good for white, negative = good for black.\nint evaluate(const Board& b);\n\n// Piece material values\nconstexpr int MAT[7] = {0, 100, 320, 330, 500, 900, 20000};\n\n// Piece-square tables (A1=0 ... H8=63, white\'s perspective).\n// For black pieces, mirror vertically: sq ^ 56\nextern const int PST[7][64];\n'

files['src/eval.cpp'] = '#include "eval.h"\n\n// Piece-square tables from white\'s perspective.\n// Layout: rank 1 at index 0-7, rank 8 at index 56-63 (A1=0 convention).\n// Positive values reward white pieces being on those squares.\nconst int PST[7][64] = {\n    // [0] unused\n    {},\n\n    // [1] Pawn\n    {\n         0,  0,  0,  0,  0,  0,  0,  0,   // rank 1 (starting row)\n         5, 10, 10,-20,-20, 10, 10,  5,   // rank 2\n         5, -5,-10,  0,  0,-10, -5,  5,   // rank 3\n         0,  0,  0, 20, 20,  0,  0,  0,   // rank 4\n         5,  5, 10, 25, 25, 10,  5,  5,   // rank 5\n        10, 10, 20, 30, 30, 20, 10, 10,   // rank 6\n        50, 50, 50, 50, 50, 50, 50, 50,   // rank 7\n         0,  0,  0,  0,  0,  0,  0,  0,   // rank 8\n    },\n\n    // [2] Knight\n    {\n        -50,-40,-30,-30,-30,-30,-40,-50,\n        -40,-20,  0,  5,  5,  0,-20,-40,\n        -30,  5, 10, 15, 15, 10,  5,-30,\n        -30,  0, 15, 20, 20, 15,  0,-30,\n        -30,  5, 15, 20, 20, 15,  5,-30,\n        -30,  0, 10, 15, 15, 10,  0,-30,\n        -40,-20,  0,  0,  0,  0,-20,-40,\n        -50,-40,-30,-30,-30,-30,-40,-50,\n    },\n\n    // [3] Bishop\n    {\n        -20,-10,-10,-10,-10,-10,-10,-20,\n        -10,  5,  0,  0,  0,  0,  5,-10,\n        -10, 10, 10, 10, 10, 10, 10,-10,\n        -10,  0, 10, 10, 10, 10,  0,-10,\n        -10,  5,  5, 10, 10,  5,  5,-10,\n        -10,  0,  5, 10, 10,  5,  0,-10,\n        -10,  0,  0,  0,  0,  0,  0,-10,\n        -20,-10,-10,-10,-10,-10,-10,-20,\n    },\n\n    // [4] Rook\n    {\n          0,  0,  0,  5,  5,  0,  0,  0,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n          5, 10, 10, 10, 10, 10, 10,  5,\n          0,  0,  0,  0,  0,  0,  0,  0,\n    },\n\n    // [5] Queen\n    {\n        -20,-10,-10, -5, -5,-10,-10,-20,\n        -10,  0,  5,  0,  0,  0,  0,-10,\n        -10,  5,  5,  5,  5,  5,  0,-10,\n          0,  0,  5,  5,  5,  5,  0, -5,\n         -5,  0,  5,  5,  5,  5,  0, -5,\n        -10,  0,  5,  5,  5,  5,  0,-10,\n        -10,  0,  0,  0,  0,  0,  0,-10,\n        -20,-10,-10, -5, -5,-10,-10,-20,\n    },\n\n    // [6] King (middlegame: hide behind pawns)\n    {\n         20, 30, 10,  0,  0, 10, 30, 20,\n         20, 20,  0,  0,  0,  0, 20, 20,\n        -10,-20,-20,-20,-20,-20,-20,-10,\n        -20,-30,-30,-40,-40,-30,-30,-20,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n    },\n};\n\nint evaluate(const Board& b) {\n    int score = 0;\n    for (int s = 0; s < 64; s++) {\n        int8_t p = b.sq[s];\n        if (p == EMPTY) continue;\n\n        int pt  = piece_type(p);\n        int pst_sq = (p > 0) ? s : (s ^ 56); // mirror for black\n        int val = MAT[pt] + PST[pt][pst_sq];\n\n        score += (p > 0) ? val : -val;\n    }\n    return score; // positive = good for white\n}\n'

files['src/search.h'] = '#pragma once\n#include "board.h"\n#include "movegen.h"\n#include <string>\n\nconstexpr int SCORE_INF  = 1\'000\'000;\nconstexpr int SCORE_MATE = 900\'000;   // checkmate score; subtract ply for faster mates\n\nstruct SearchResult {\n    Move      best{Move::null()};\n    int       score{0};\n    long long nodes{0};\n};\n\n// Serial negamax + alpha-beta pruning\nSearchResult search_serial(Board& board, int depth);\n\n// OpenMP root-parallel search (requires -DUSE_OMP and OpenMP)\n#ifdef USE_OMP\nSearchResult search_omp(Board& board, int depth);\n#endif\n\n// CUDA batch-evaluation search (requires -DUSE_CUDA and nvcc)\n#ifdef USE_CUDA\nSearchResult search_cuda(Board& board, int depth);\n#endif\n'

files['src/search_serial.cpp'] = '#include "search.h"\n#include "eval.h"\n#include <algorithm>\n#include <vector>\n\n// ── Move ordering ─────────────────────────────────────────────────────────\n// MVV-LVA: most-valuable-victim / least-valuable-attacker heuristic.\n// Captures are sorted before quiet moves; among captures, prefer taking\n// high-value pieces with low-value attackers.\nstatic int move_score(const Move& m, const Board& b) {\n    if (m.flags & Move::CAPTURE) {\n        int victim   = MAT[piece_type(b.sq[m.to])];\n        int attacker = MAT[piece_type(b.sq[m.from])];\n        return 10000 + victim - attacker / 10;\n    }\n    return 0;\n}\n\nstatic void order_moves(std::vector<Move>& moves, const Board& b) {\n    std::stable_sort(moves.begin(), moves.end(), [&](const Move& a, const Move& bm) {\n        return move_score(a, b) > move_score(bm, b);\n    });\n}\n\n// ── Quiescence search ─────────────────────────────────────────────────────\n// Extend search until the position is "quiet" (no captures left), preventing\n// the horizon effect where evaluation cuts off mid-exchange.\nstatic int quiesce(Board& b, int alpha, int beta, long long& nodes) {\n    nodes++;\n\n    // Static evaluation from current side\'s perspective\n    int stand_pat = evaluate(b);\n    if (!b.wtm) stand_pat = -stand_pat;\n\n    if (stand_pat >= beta) return beta;\n    if (stand_pat > alpha) alpha = stand_pat;\n\n    auto moves = generate_pseudo_legal(b);\n    order_moves(moves, b);\n\n    for (const auto& m : moves) {\n        if (!(m.flags & Move::CAPTURE)) continue;\n\n        auto u = b.make_move(m);\n        bool legal = !b.square_attacked(b.king_sq(!b.wtm), b.wtm);\n        int score = 0;\n        if (legal) score = -quiesce(b, -beta, -alpha, nodes);\n        b.undo_move(m, u);\n\n        if (!legal) continue;\n        if (score >= beta) return beta;\n        if (score > alpha) alpha = score;\n    }\n    return alpha;\n}\n\n// ── Negamax with alpha-beta pruning ───────────────────────────────────────\n// Returns a score from the current side\'s perspective.\n// depth = remaining plies; ply = distance from root (for mate scoring).\nint negamax(Board& b, int depth, int ply, int alpha, int beta, long long& nodes) {\n    nodes++;\n\n    if (depth == 0)\n        return quiesce(b, alpha, beta, nodes);\n\n    auto moves = generate_legal_moves(b);\n\n    if (moves.empty())\n        return b.in_check() ? -(SCORE_MATE - ply) : 0; // checkmate or stalemate\n\n    order_moves(moves, b);\n\n    for (const auto& m : moves) {\n        auto u = b.make_move(m);\n        int score = -negamax(b, depth - 1, ply + 1, -beta, -alpha, nodes);\n        b.undo_move(m, u);\n\n        if (score >= beta) return beta; // beta cutoff (fail-high)\n        if (score > alpha) alpha = score;\n    }\n    return alpha;\n}\n\n// ── Root search ───────────────────────────────────────────────────────────\nSearchResult search_serial(Board& board, int depth) {\n    auto moves = generate_legal_moves(board);\n    if (moves.empty()) return {};\n\n    order_moves(moves, board);\n\n    SearchResult res;\n    res.best  = moves[0];\n    int alpha = -SCORE_INF;\n\n    for (const auto& m : moves) {\n        auto u = board.make_move(m);\n        int score = -negamax(board, depth - 1, 1, -SCORE_INF, -alpha, res.nodes);\n        board.undo_move(m, u);\n\n        if (score > alpha) {\n            alpha    = score;\n            res.best = m;\n        }\n    }\n    res.score = alpha;\n    return res;\n}\n'

files['src/search_omp.cpp'] = '#ifdef USE_OMP\n#include "search.h"\n#include "eval.h"\n#ifdef __APPLE__\n#include "/opt/homebrew/opt/libomp/include/omp.h"\n#else\n#include <omp.h>\n#endif\n#include <algorithm>\n#include <atomic>\n#include <vector>\n\n// Shared declaration from search_serial.cpp\nint negamax(Board& b, int depth, int ply, int alpha, int beta, long long& nodes);\n\nstatic int move_score(const Move& m, const Board& b) {\n    if (m.flags & Move::CAPTURE) {\n        int victim   = MAT[piece_type(b.sq[m.to])];\n        int attacker = MAT[piece_type(b.sq[m.from])];\n        return 10000 + victim - attacker / 10;\n    }\n    return 0;\n}\n\nstatic void order_moves(std::vector<Move>& moves, const Board& b) {\n    std::stable_sort(moves.begin(), moves.end(), [&](const Move& a, const Move& bm) {\n        return move_score(a, b) > move_score(bm, b);\n    });\n}\n\n// ── OpenMP root-parallel search ───────────────────────────────────────────\n//\n// Strategy:\n//   1. Evaluate the first (best-ordered) root move serially to get a tight\n//      alpha lower bound before spawning threads. This dramatically improves\n//      beta-pruning efficiency in the parallel phase.\n//   2. Remaining moves are split across threads. Each thread reads the\n//      shared alpha atomically; completed threads publish their score so\n//      subsequent threads start with a tighter window.\nSearchResult search_omp(Board& board, int depth) {\n    auto moves = generate_legal_moves(board);\n    if (moves.empty()) return {};\n\n    order_moves(moves, board);\n\n    const int n = (int)moves.size();\n    SearchResult res;\n    res.best = moves[0];\n\n    // Phase 1: search best move serially to establish alpha\n    {\n        auto u    = board.make_move(moves[0]);\n        int score = -negamax(board, depth - 1, 1, -SCORE_INF, SCORE_INF, res.nodes);\n        board.undo_move(moves[0], u);\n        res.score = score;\n    }\n\n    if (n == 1) return res;\n\n    std::atomic<int>       global_alpha{res.score};\n    int                    best_idx = 0;\n    std::atomic<long long> total_nodes{res.nodes};\n\n    // Phase 2: parallelize remaining moves with alpha = best score so far\n    #pragma omp parallel for schedule(dynamic, 1)\n    for (int i = 1; i < n; i++) {\n        Board     local = board;\n        long long nodes = 0;\n\n        int  alpha = global_alpha.load(std::memory_order_relaxed);\n        auto u     = local.make_move(moves[i]);\n        int  score = -negamax(local, depth - 1, 1, -SCORE_INF, -alpha, nodes);\n        local.undo_move(moves[i], u);\n\n        total_nodes.fetch_add(nodes, std::memory_order_relaxed);\n\n        #pragma omp critical\n        {\n            if (score > global_alpha.load()) {\n                global_alpha.store(score);\n                best_idx = i;\n            }\n        }\n    }\n\n    int final_alpha = global_alpha.load();\n    if (final_alpha > res.score) {\n        res.score = final_alpha;\n        res.best  = moves[best_idx];\n    }\n    res.nodes = total_nodes.load();\n    return res;\n}\n\n#endif // USE_OMP\n'

files['src/search_cuda.cpp'] = '#ifdef USE_CUDA\n// search_cuda.cpp — Hybrid CPU/GPU search\n//\n// Strategy:\n//   Alpha-beta runs on the CPU down to depth N-1. At depth 1 (one ply before\n//   leaves), instead of calling negamax recursively, we generate ALL child\n//   positions, batch-evaluate them on the GPU in one kernel launch, and feed\n//   the GPU scores back into the minimax. This gives the GPU thousands of\n//   positions to evaluate in parallel.\n\n#include "search.h"\n#include "eval.h"\n#include "../cuda/eval_cuda.cuh"\n#include <algorithm>\n#include <vector>\n\nstatic int move_score(const Move& m, const Board& b) {\n    if (m.flags & Move::CAPTURE) {\n        int victim   = MAT[piece_type(b.sq[m.to])];\n        int attacker = MAT[piece_type(b.sq[m.from])];\n        return 10000 + victim - attacker / 10;\n    }\n    return 0;\n}\n\nstatic void order_moves(std::vector<Move>& moves, const Board& b) {\n    std::stable_sort(moves.begin(), moves.end(), [&](const Move& a, const Move& bm) {\n        return move_score(a, b) > move_score(bm, b);\n    });\n}\n\n// Shared from search_serial.cpp\nint negamax(Board& b, int depth, int ply, int alpha, int beta, long long& nodes);\n\n// At depth 1: collect all child positions, batch-evaluate on GPU,\n// return the best score from the current side\'s perspective.\nstatic int gpu_leaf(Board& b, int ply, int alpha, int beta, long long& nodes) {\n    auto moves = generate_legal_moves(b);\n    if (moves.empty())\n        return b.in_check() ? -(SCORE_MATE - ply) : 0;\n\n    order_moves(moves, b);\n\n    // Build batch of child positions\n    std::vector<BoardData> batch;\n    batch.reserve(moves.size());\n    std::vector<bool> legal_flags(moves.size(), false);\n\n    for (int i = 0; i < (int)moves.size(); i++) {\n        auto u = b.make_move(moves[i]);\n        // Position is always legal here (generate_legal_moves already filtered)\n        BoardData bd;\n        std::copy(b.sq, b.sq + 64, bd.sq);\n        bd.wtm = b.wtm ? 1 : -1;\n        batch.push_back(bd);\n        legal_flags[i] = true;\n        b.undo_move(moves[i], u);\n    }\n\n    nodes += (long long)batch.size();\n\n    // GPU evaluates all child positions in parallel\n    auto gpu_scores = gpu_evaluate_batch(batch);\n\n    // Minimax over GPU scores\n    int best = -SCORE_INF;\n    for (int i = 0; i < (int)moves.size(); i++) {\n        if (!legal_flags[i]) continue;\n        // gpu_scores[i] is from the child\'s side perspective; negate for parent\n        int score = -gpu_scores[i];\n        if (score >= beta) return beta;\n        if (score > best) best = score;\n        if (score > alpha) alpha = score;\n    }\n    return best;\n}\n\n// Hybrid negamax: CPU alpha-beta at higher depths, GPU batch at depth == 1\nstatic int negamax_cuda(Board& b, int depth, int ply, int alpha, int beta, long long& nodes) {\n    nodes++;\n\n    if (depth == 1)\n        return gpu_leaf(b, ply, alpha, beta, nodes);\n\n    if (depth == 0) {\n        // Quiescence not wrapped in GPU here; fall back to CPU evaluate\n        int score = evaluate(b);\n        return b.wtm ? score : -score;\n    }\n\n    auto moves = generate_legal_moves(b);\n    if (moves.empty())\n        return b.in_check() ? -(SCORE_MATE - ply) : 0;\n\n    order_moves(moves, b);\n\n    for (const auto& m : moves) {\n        auto u = b.make_move(m);\n        int score = -negamax_cuda(b, depth - 1, ply + 1, -beta, -alpha, nodes);\n        b.undo_move(m, u);\n\n        if (score >= beta) return beta;\n        if (score > alpha) alpha = score;\n    }\n    return alpha;\n}\n\n// ── Root search ───────────────────────────────────────────────────────────\nSearchResult search_cuda(Board& board, int depth) {\n    cuda_init();\n\n    auto moves = generate_legal_moves(board);\n    if (moves.empty()) return {};\n\n    order_moves(moves, board);\n\n    SearchResult res;\n    res.best  = moves[0];\n    int alpha = -SCORE_INF;\n\n    for (const auto& m : moves) {\n        auto u = board.make_move(m);\n        int score = -negamax_cuda(board, depth - 1, 1, -SCORE_INF, -alpha, res.nodes);\n        board.undo_move(m, u);\n\n        if (score > alpha) {\n            alpha    = score;\n            res.best = m;\n        }\n    }\n    res.score = alpha;\n    return res;\n}\n\n#endif // USE_CUDA\n'

files['src/main.cpp'] = '// main.cpp — Chess engine driver\n//\n// Modes:\n//   bench  [--fen FEN] [--depth N]  — benchmark serial / OpenMP / CUDA on one position\n//   report [--fen FEN] [--maxdepth N] — sweep depths, print table, write perf_report.html\n//   perft  [--fen FEN] [--depth N]  — move-generation correctness test (node counts)\n//   play   [--fen FEN] [--depth N]  — engine plays from position (prints best move)\n//   uci                             — minimal UCI loop\n\n#include <algorithm>\n#include <chrono>\n#include <cstdio>\n#include <cstring>\n#include <fstream>\n#include <iostream>\n#include <sstream>\n#include <string>\n#include <vector>\n\n#include "board.h"\n#include "eval.h"\n#include "movegen.h"\n#include "search.h"\n\n#ifdef USE_OMP\n#ifdef __APPLE__\n#include "/opt/homebrew/opt/libomp/include/omp.h"\n#else\n#include <omp.h>\n#endif\n#endif\n\n#ifdef USE_CUDA\n#include "../cuda/eval_cuda.cuh"\n#endif\n\nusing Clock = std::chrono::high_resolution_clock;\ninline double elapsed_ms(Clock::time_point t0) {\n    return std::chrono::duration<double, std::milli>(Clock::now() - t0).count();\n}\n\n// ── Perft ─────────────────────────────────────────────────────────────────\n\nstatic long long perft(Board& b, int depth) {\n    if (depth == 0) return 1;\n    auto moves = generate_legal_moves(b);\n    if (depth == 1) return (long long)moves.size();\n    long long total = 0;\n    for (const auto& m : moves) {\n        auto u = b.make_move(m);\n        total += perft(b, depth - 1);\n        b.undo_move(m, u);\n    }\n    return total;\n}\n\nstatic void run_perft(const std::string& fen, int depth) {\n    Board b;\n    b.load_fen(fen);\n    b.print();\n    printf("\\nPerft depth %d:\\n", depth);\n    auto  moves = generate_legal_moves(b);\n    long long total = 0;\n    auto t0 = Clock::now();\n    for (const auto& m : moves) {\n        auto u   = b.make_move(m);\n        long long cnt = perft(b, depth - 1);\n        b.undo_move(m, u);\n        printf("  %s: %lld\\n", move_to_str(m).c_str(), cnt);\n        total += cnt;\n    }\n    double ms = elapsed_ms(t0);\n    printf("\\nTotal: %lld  Time: %.1f ms  NPS: %.0f\\n", total, ms, total / (ms / 1000.0));\n}\n\n// ── Single-depth benchmark ────────────────────────────────────────────────\n\nstatic void print_result(const char* tag, const SearchResult& r, double ms) {\n    printf("  [%-6s]  move: %-6s  score: %+6d  nodes: %8lld  time: %8.1f ms  knps: %6.0f\\n",\n           tag, move_to_str(r.best).c_str(), r.score, r.nodes, ms, r.nodes / ms);\n}\n\nstatic void run_bench(const std::string& fen, int depth) {\n    Board b;\n    b.load_fen(fen);\n    b.print();\n    printf("\\nSearching depth %d ...\\n\\n", depth);\n\n    { Board l = b; auto t = Clock::now(); auto r = search_serial(l, depth); print_result("serial", r, elapsed_ms(t)); }\n#ifdef USE_OMP\n    { Board l = b; auto t = Clock::now(); auto r = search_omp(l, depth);    print_result("openmp", r, elapsed_ms(t)); }\n#endif\n#ifdef USE_CUDA\n    { cuda_init(); Board l = b; auto t = Clock::now(); auto r = search_cuda(l, depth); print_result("cuda", r, elapsed_ms(t)); cuda_cleanup(); }\n#endif\n}\n\n// ── Multi-depth report ────────────────────────────────────────────────────\n\nstruct DepthRow {\n    int       depth;\n    double    serial_ms;\n    long long serial_nodes;\n    double    omp_ms;\n    long long omp_nodes;\n    double    cuda_ms;\n    long long cuda_nodes;\n};\n\nstatic void write_html_report(const std::vector<DepthRow>& rows,\n                               const std::string& fen,\n                               const std::string& outfile) {\n    const bool has_cuda = std::any_of(\n        rows.begin(), rows.end(), [](const DepthRow& row) { return row.cuda_ms > 0.0; }\n    );\n\n    // Build JS arrays\n    std::string labels, s_ms, o_ms, c_ms, speedup_omp, speedup_cuda, s_knps, o_knps, c_knps;\n    for (int i = 0; i < (int)rows.size(); i++) {\n        const auto& r = rows[i];\n        if (i) {\n            labels += ",";\n            s_ms += ",";\n            o_ms += ",";\n            c_ms += ",";\n            speedup_omp += ",";\n            speedup_cuda += ",";\n            s_knps += ",";\n            o_knps += ",";\n            c_knps += ",";\n        }\n        labels       += std::to_string(r.depth);\n        s_ms         += std::to_string((long long)r.serial_ms);\n        o_ms         += std::to_string((long long)r.omp_ms);\n        c_ms         += r.cuda_ms > 0 ? std::to_string((long long)r.cuda_ms) : "null";\n        speedup_omp  += std::to_string(r.serial_ms / r.omp_ms).substr(0, 5);\n        speedup_cuda += r.cuda_ms > 0 ? std::to_string(r.serial_ms / r.cuda_ms).substr(0, 5) : "null";\n        s_knps       += std::to_string((long long)(r.serial_nodes / r.serial_ms));\n        o_knps       += std::to_string((long long)(r.omp_nodes / r.omp_ms));\n        c_knps       += r.cuda_ms > 0 ? std::to_string((long long)(r.cuda_nodes / r.cuda_ms)) : "null";\n    }\n\n    // Build ASCII table rows for the HTML table\n    std::string table_rows;\n    for (const auto& r : rows) {\n        double sup_omp  = r.serial_ms / r.omp_ms;\n        double sup_cuda = r.cuda_ms > 0 ? r.serial_ms / r.cuda_ms : 0.0;\n        table_rows += "<tr>";\n        table_rows += "<td>" + std::to_string(r.depth) + "</td>";\n        table_rows += "<td>" + std::to_string(r.serial_ms).substr(0, std::to_string(r.serial_ms).find(\'.\') + 2) + "</td>";\n        table_rows += "<td>" + std::to_string(r.serial_nodes) + "</td>";\n        table_rows += "<td>" + std::to_string((long long)(r.serial_nodes / r.serial_ms)) + "</td>";\n        table_rows += "<td>" + std::to_string(r.omp_ms).substr(0, std::to_string(r.omp_ms).find(\'.\') + 2) + "</td>";\n        table_rows += "<td>" + std::to_string(r.omp_nodes) + "</td>";\n        table_rows += "<td>" + std::to_string(sup_omp).substr(0, std::to_string(sup_omp).find(\'.\') + 3) + "×</td>";\n        if (has_cuda) {\n            if (sup_cuda > 0) {\n                table_rows += "<td>" + std::to_string(r.cuda_ms).substr(0, std::to_string(r.cuda_ms).find(\'.\') + 2) + "</td>";\n                table_rows += "<td>" + std::to_string(r.cuda_nodes) + "</td>";\n                table_rows += "<td>" + std::to_string(sup_cuda).substr(0, std::to_string(sup_cuda).find(\'.\') + 3) + "×</td>";\n            } else {\n                table_rows += "<td>n/a</td><td>n/a</td><td>n/a</td>";\n            }\n        }\n        table_rows += "</tr>\\n";\n    }\n\n    std::ofstream f(outfile);\n    f << R"(<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<title>Chess Engine Performance Report</title>\n<script src="https://cdn.jsdelivr.net/npm/chart.js@4"></script>\n<style>\n  body { font-family: \'Segoe UI\', Arial, sans-serif; background: #0d1117; color: #e6edf3; margin: 0; padding: 32px; }\n  h1   { font-size: 1.8rem; margin-bottom: 4px; color: #58a6ff; }\n  .sub { color: #8b949e; font-size: .9rem; margin-bottom: 36px; font-family: monospace; }\n  .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 28px; margin-bottom: 36px; }\n  .card { background: #161b22; border: 1px solid #30363d; border-radius: 10px; padding: 24px; }\n  .card h2 { margin: 0 0 16px; font-size: 1rem; color: #8b949e; text-transform: uppercase; letter-spacing: .08em; }\n  table { width: 100%; border-collapse: collapse; font-size: .88rem; }\n  th    { background: #21262d; color: #8b949e; padding: 10px 14px; text-align: right; font-weight: 600; white-space: nowrap; }\n  th:first-child { text-align: center; }\n  td    { padding: 9px 14px; text-align: right; border-bottom: 1px solid #21262d; }\n  td:first-child { text-align: center; font-weight: 700; color: #58a6ff; }\n  tr:last-child td { border-bottom: none; }\n  .speedup { color: #3fb950; font-weight: 700; }\n  .badge   { display: inline-block; padding: 2px 8px; border-radius: 4px; font-size: .75rem; font-weight: 700; margin-left: 8px; }\n  .serial-badge { background: #1f3a5f; color: #79c0ff; }\n  .omp-badge    { background: #1f3a2f; color: #56d364; }\n  .cuda-badge   { background: #3a1f5f; color: #d2a8ff; }\n</style>\n</head>\n<body>\n<h1>Chess Engine Performance Report</h1>\n<p class="sub">Position: )";\n    f << fen << "</p>\\n";\n\n    f << R"(\n<div class="grid">\n  <div class="card">\n    <h2>Search Time (ms) — lower is better</h2>\n    <canvas id="timeChart"></canvas>\n  </div>\n  <div class="card">\n    <h2>Speedup vs Serial — higher is better</h2>\n    <canvas id="speedupChart"></canvas>\n  </div>\n  <div class="card">\n    <h2>Throughput (kilo-nodes/ms) — higher is better</h2>\n    <canvas id="knpsChart"></canvas>\n  </div>\n  <div class="card">\n    <h2>Search Time — log scale</h2>\n    <canvas id="logChart"></canvas>\n  </div>\n</div>\n\n<div class="card" style="margin-bottom:36px">\n  <h2>Raw Data</h2>\n  <table>\n    <thead>\n      <tr>\n        <th rowspan="2">Depth</th>\n        <th colspan="3"><span class="badge serial-badge">SERIAL</span></th>\n        <th colspan="3"><span class="badge omp-badge">OPENMP</span></th>\n)";\n    if (has_cuda) {\n        f << R"(\n        <th colspan="3"><span class="badge cuda-badge">CUDA</span></th>\n)";\n    }\n    f << R"(\n      </tr>\n      <tr>\n        <th>Time (ms)</th><th>Nodes</th><th>kn/ms</th>\n        <th>Time (ms)</th><th>Nodes</th><th>Speedup</th>)";\n    if (has_cuda) {\n        f << R"(<th>Time (ms)</th><th>Nodes</th><th>Speedup</th>)";\n    }\n    f << R"(\n      </tr>\n    </thead>\n    <tbody>\n)";\n    f << table_rows;\n    f << R"(    </tbody>\n  </table>\n</div>\n\n<script>\nconst labels = [)" << labels << R"(];\nconst cfg = (type, datasets, extra={}) => ({\n  type, data: { labels, datasets },\n  options: {\n    responsive: true,\n    plugins: { legend: { labels: { color: \'#8b949e\' } } },\n    scales: {\n      x: { ticks: { color: \'#8b949e\' }, grid: { color: \'#21262d\' }, title: { display: true, text: \'Search Depth\', color: \'#8b949e\' } },\n      y: { ticks: { color: \'#8b949e\' }, grid: { color: \'#21262d\' }, ...extra }\n    }\n  }\n});\n\n// Time chart (bar)\nnew Chart(document.getElementById(\'timeChart\'), cfg(\'bar\', [\n  { label: \'Serial\', data: [)" << s_ms << R"(], backgroundColor: \'rgba(88,166,255,0.7)\', borderColor: \'#58a6ff\', borderWidth: 1 },\n  { label: \'OpenMP\', data: [)" << o_ms << R"(], backgroundColor: \'rgba(63,185,80,0.7)\',  borderColor: \'#3fb950\', borderWidth: 1 })";\n    if (has_cuda) {\n        f << R"(,\n  { label: \'CUDA\',   data: [)" << c_ms << R"(], backgroundColor: \'rgba(188,140,255,0.7)\',borderColor: \'#bc8cff\', borderWidth: 1 })";\n    }\n    f << R"(\n]));\n\n// Speedup chart (line)\nnew Chart(document.getElementById(\'speedupChart\'), cfg(\'line\', [\n  { label: \'OpenMP speedup\', data: [)" << speedup_omp << R"(], borderColor: \'#3fb950\', backgroundColor: \'rgba(63,185,80,0.15)\', tension: 0.3, fill: true })";\n    if (has_cuda) {\n        f << R"(,\n  { label: \'CUDA speedup\',   data: [)" << speedup_cuda << R"(], borderColor: \'#bc8cff\', backgroundColor: \'rgba(188,140,255,0.15)\', tension: 0.3, fill: true })";\n    }\n    f << R"(\n], { title: { display: true, text: \'× faster than serial\', color: \'#8b949e\' } }));\n\n// knps chart\nnew Chart(document.getElementById(\'knpsChart\'), cfg(\'bar\', [\n  { label: \'Serial\',  data: [)" << s_knps << R"(], backgroundColor: \'rgba(88,166,255,0.7)\',  borderColor: \'#58a6ff\', borderWidth: 1 },\n  { label: \'OpenMP\',  data: [)" << o_knps << R"(], backgroundColor: \'rgba(63,185,80,0.7)\',   borderColor: \'#3fb950\', borderWidth: 1 },\n)";\n    if (has_cuda) {\n        f << R"(\n  { label: \'CUDA\',    data: [)" << c_knps << R"(], backgroundColor: \'rgba(188,140,255,0.7)\', borderColor: \'#bc8cff\', borderWidth: 1 },\n)";\n    }\n    f << R"(\n]));\n\n// Log-scale time chart\nnew Chart(document.getElementById(\'logChart\'), cfg(\'line\', [\n  { label: \'Serial\', data: [)" << s_ms << R"(], borderColor: \'#58a6ff\', tension: 0.3, fill: false },\n  { label: \'OpenMP\', data: [)" << o_ms << R"(], borderColor: \'#3fb950\', tension: 0.3, fill: false })";\n    if (has_cuda) {\n        f << R"(,\n  { label: \'CUDA\',   data: [)" << c_ms << R"(], borderColor: \'#bc8cff\', tension: 0.3, fill: false })";\n    }\n    f << R"(\n], { type: \'logarithmic\', title: { display: true, text: \'ms (log)\', color: \'#8b949e\' } }));\n</script>\n</body>\n</html>\n)";\n    f.close();\n    printf("\\nReport written to: %s\\n", outfile.c_str());\n}\n\nstatic void run_report(const std::string& fen, int maxdepth, const std::string& outfile) {\n    Board b;\n    b.load_fen(fen);\n    b.print();\n\n    int ncores = 1;\n#ifdef USE_OMP\n    #pragma omp parallel\n    { ncores = omp_get_num_threads(); }\n#endif\n\n    printf("\\nSweeping depths 3..%d | position: %s\\n", maxdepth, fen.c_str());\n    printf("CPU threads: %d\\n\\n", ncores);\n\n    // Header\n    printf("%-5s  %-28s  %-28s  %s\\n",\n           "Depth",\n           "──── Serial ────────────────",\n           "──── OpenMP ───────────────",\n           "Speedup");\n    printf("%-5s  %-10s %-10s %-6s  %-10s %-10s %-6s  %s\\n",\n           "", "Time(ms)", "Nodes", "kn/ms",\n               "Time(ms)", "Nodes", "kn/ms", "OMP");\n    printf("%s\\n", std::string(80, \'-\').c_str());\n\n    std::vector<DepthRow> rows;\n\n    for (int d = 3; d <= maxdepth; d++) {\n        DepthRow row{};\n        row.depth = d;\n\n        // Serial\n        {\n            Board l = b;\n            auto  t = Clock::now();\n            auto  r = search_serial(l, d);\n            row.serial_ms    = elapsed_ms(t);\n            row.serial_nodes = r.nodes;\n        }\n\n#ifdef USE_OMP\n        // OpenMP\n        {\n            Board l = b;\n            auto  t = Clock::now();\n            auto  r = search_omp(l, d);\n            row.omp_ms    = elapsed_ms(t);\n            row.omp_nodes = r.nodes;\n        }\n#else\n        row.omp_ms = row.serial_ms; row.omp_nodes = row.serial_nodes;\n#endif\n\n#ifdef USE_CUDA\n        {\n            cuda_init();\n            Board l = b;\n            auto  t = Clock::now();\n            auto  r = search_cuda(l, d);\n            row.cuda_ms    = elapsed_ms(t);\n            row.cuda_nodes = r.nodes;\n            cuda_cleanup();\n        }\n#endif\n\n        double sup_omp  = row.serial_ms / row.omp_ms;\n        double sup_cuda = row.cuda_ms > 0 ? row.serial_ms / row.cuda_ms : 0.0;\n\n        printf("%-5d  %10.1f %10lld %6lld  %10.1f %10lld %6lld  %5.2f×",\n               d,\n               row.serial_ms, row.serial_nodes, (long long)(row.serial_nodes / row.serial_ms),\n               row.omp_ms,    row.omp_nodes,    (long long)(row.omp_nodes / row.omp_ms),\n               sup_omp);\n        if (sup_cuda > 0) printf("  CUDA %.2f×", sup_cuda);\n        printf("\\n");\n        fflush(stdout);\n\n        rows.push_back(row);\n    }\n\n    printf("%s\\n", std::string(80, \'-\').c_str());\n    write_html_report(rows, fen, outfile);\n}\n\n// ── Minimal UCI loop ──────────────────────────────────────────────────────\n\nstatic void run_uci(int default_depth) {\n    printf("id name CudaChess\\nid author gurnoor\\nuciok\\n");\n    Board board; board.set_startpos();\n    int depth = default_depth;\n    std::string line;\n    while (std::getline(std::cin, line)) {\n        std::istringstream ss(line); std::string tok; ss >> tok;\n        if (tok == "quit") break;\n        if (tok == "isready") { printf("readyok\\n"); }\n        else if (tok == "ucinewgame") { board.set_startpos(); }\n        else if (tok == "position") {\n            ss >> tok;\n            if (tok == "startpos") { board.set_startpos(); ss >> tok; }\n            else if (tok == "fen") {\n                std::string fen;\n                for (int i = 0; i < 6 && ss >> tok; i++) fen += (i ? " " : "") + tok;\n                board.load_fen(fen); ss >> tok;\n            }\n            while (ss >> tok) {\n                auto moves = generate_legal_moves(board);\n                for (const auto& m : moves) if (move_to_str(m) == tok) { board.make_move(m); break; }\n            }\n        } else if (tok == "go") {\n            std::string opt;\n            while (ss >> opt) if (opt == "depth") ss >> depth;\n            auto res = search_serial(board, depth);\n            printf("bestmove %s\\n", move_to_str(res.best).c_str()); fflush(stdout);\n        } else if (tok == "d") { board.print(); }\n    }\n}\n\n// ── Entry point ───────────────────────────────────────────────────────────\n\nstatic const char* STARTFEN  = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1";\nstatic const char* BENCH_FEN = "r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1";\n\nint main(int argc, char** argv) {\n    std::string mode     = "bench";\n    std::string fen      = BENCH_FEN;\n    std::string outfile  = "perf_report.html";\n    int         depth    = 6;\n    int         maxdepth = 8;\n\n    for (int i = 1; i < argc; i++) {\n        std::string a = argv[i];\n        if (a == "bench" || a == "perft" || a == "play" || a == "uci" || a == "report") {\n            mode = a;\n        } else if (a == "--fen"      && i+1 < argc) { fen      = argv[++i]; }\n          else if (a == "--depth"    && i+1 < argc) { depth    = std::atoi(argv[++i]); }\n          else if (a == "--maxdepth" && i+1 < argc) { maxdepth = std::atoi(argv[++i]); }\n          else if (a == "--out"      && i+1 < argc) { outfile  = argv[++i]; }\n          else if (a == "--startpos")               { fen      = STARTFEN; }\n    }\n\n    if      (mode == "uci")    { run_uci(depth); }\n    else if (mode == "perft")  { run_perft(fen, depth); }\n    else if (mode == "play")   {\n        Board b; b.load_fen(fen); b.print();\n        auto r = search_serial(b, depth);\n        printf("\\nBest: %s  Score: %+d  Nodes: %lld\\n",\n               move_to_str(r.best).c_str(), r.score, r.nodes);\n    }\n    else if (mode == "report") { run_report(fen, maxdepth, outfile); }\n    else                       { run_bench(fen, depth); }\n\n    return 0;\n}\n'

files['cuda/eval_cuda.cuh'] = "#pragma once\n#include <cstdint>\n#include <vector>\n\n// Compact board state for GPU transfer (64 bytes + 1 byte side-to-move)\nstruct BoardData {\n    int8_t sq[64];\n    int8_t wtm; // 1 = white to move, -1 = black to move\n};\n\n// Evaluate a batch of board positions on the GPU.\n// Returns a vector of scores (centipawns) from each position's current\n// side-to-move perspective (positive = good for the side to move).\nstd::vector<int> gpu_evaluate_batch(const std::vector<BoardData>& positions);\n\n// Call once at program start / end if CUDA is available\nvoid cuda_init();\nvoid cuda_cleanup();\n"

files['cuda/eval_cuda.cu'] = '// eval_cuda.cu — GPU batch evaluation of chess positions\n//\n// Architecture:\n//   Each CUDA thread evaluates one board position using material + PST scores.\n//   The CPU hands off a flat array of board states; the GPU evaluates them all\n//   in parallel and returns an array of integer scores.\n//\n// This file is compiled with nvcc; everything else with g++/clang.\n\n#include "eval_cuda.cuh"\n#include <cuda_runtime.h>\n#include <cstdio>\n#include <cstring>\n#include <stdexcept>\n#include <string>\n\n#define CUDA_CHECK(call) \\\n    do { \\\n        cudaError_t _e = (call); \\\n        if (_e != cudaSuccess) \\\n            throw std::runtime_error(std::string("CUDA error: ") + cudaGetErrorString(_e)); \\\n    } while (0)\n\n// ── Device-side constants ─────────────────────────────────────────────────\n// Stored in fast __constant__ memory (broadcast to all threads).\n\n__constant__ int d_MAT[7] = {0, 100, 320, 330, 500, 900, 20000};\n\n// PST[piece_type][square], A1=0 ... H8=63, white\'s perspective.\n// For a black piece at square s, index with (s ^ 56) to mirror vertically.\n__constant__ int d_PST[7][64] = {\n    // [0] unused\n    {},\n    // [1] Pawn\n    {\n         0,  0,  0,  0,  0,  0,  0,  0,\n         5, 10, 10,-20,-20, 10, 10,  5,\n         5, -5,-10,  0,  0,-10, -5,  5,\n         0,  0,  0, 20, 20,  0,  0,  0,\n         5,  5, 10, 25, 25, 10,  5,  5,\n        10, 10, 20, 30, 30, 20, 10, 10,\n        50, 50, 50, 50, 50, 50, 50, 50,\n         0,  0,  0,  0,  0,  0,  0,  0,\n    },\n    // [2] Knight\n    {\n        -50,-40,-30,-30,-30,-30,-40,-50,\n        -40,-20,  0,  5,  5,  0,-20,-40,\n        -30,  5, 10, 15, 15, 10,  5,-30,\n        -30,  0, 15, 20, 20, 15,  0,-30,\n        -30,  5, 15, 20, 20, 15,  5,-30,\n        -30,  0, 10, 15, 15, 10,  0,-30,\n        -40,-20,  0,  0,  0,  0,-20,-40,\n        -50,-40,-30,-30,-30,-30,-40,-50,\n    },\n    // [3] Bishop\n    {\n        -20,-10,-10,-10,-10,-10,-10,-20,\n        -10,  5,  0,  0,  0,  0,  5,-10,\n        -10, 10, 10, 10, 10, 10, 10,-10,\n        -10,  0, 10, 10, 10, 10,  0,-10,\n        -10,  5,  5, 10, 10,  5,  5,-10,\n        -10,  0,  5, 10, 10,  5,  0,-10,\n        -10,  0,  0,  0,  0,  0,  0,-10,\n        -20,-10,-10,-10,-10,-10,-10,-20,\n    },\n    // [4] Rook\n    {\n          0,  0,  0,  5,  5,  0,  0,  0,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n         -5,  0,  0,  0,  0,  0,  0, -5,\n          5, 10, 10, 10, 10, 10, 10,  5,\n          0,  0,  0,  0,  0,  0,  0,  0,\n    },\n    // [5] Queen\n    {\n        -20,-10,-10, -5, -5,-10,-10,-20,\n        -10,  0,  5,  0,  0,  0,  0,-10,\n        -10,  5,  5,  5,  5,  5,  0,-10,\n          0,  0,  5,  5,  5,  5,  0, -5,\n         -5,  0,  5,  5,  5,  5,  0, -5,\n        -10,  0,  5,  5,  5,  5,  0,-10,\n        -10,  0,  0,  0,  0,  0,  0,-10,\n        -20,-10,-10, -5, -5,-10,-10,-20,\n    },\n    // [6] King (middlegame)\n    {\n         20, 30, 10,  0,  0, 10, 30, 20,\n         20, 20,  0,  0,  0,  0, 20, 20,\n        -10,-20,-20,-20,-20,-20,-20,-10,\n        -20,-30,-30,-40,-40,-30,-30,-20,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n        -30,-40,-40,-50,-50,-40,-40,-30,\n    },\n};\n\n// ── GPU evaluation kernel ─────────────────────────────────────────────────\n//\n// Each thread evaluates ONE board position.\n// boards:   flat array of (n * 64) int8_t — board states packed back-to-back\n// wtm_arr:  n int8_t — side to move (1=white, -1=black)\n// scores:   n int    — output scores (current side\'s perspective)\n// n:        number of positions\n__global__ void eval_kernel(\n    const int8_t* __restrict__ boards,\n    const int8_t* __restrict__ wtm_arr,\n    int*          __restrict__ scores,\n    int n)\n{\n    int idx = blockIdx.x * blockDim.x + threadIdx.x;\n    if (idx >= n) return;\n\n    const int8_t* board = boards + idx * 64;\n    int score = 0;\n\n    for (int s = 0; s < 64; s++) {\n        int8_t p = board[s];\n        if (p == 0) continue;\n\n        int sign   = (p > 0) ? 1 : -1;\n        int pt     = sign * p;            // piece type: 1-6\n        int pst_sq = (p > 0) ? s : (s ^ 56); // mirror for black\n\n        score += sign * (d_MAT[pt] + d_PST[pt][pst_sq]);\n    }\n\n    // Convert to current-side perspective\n    scores[idx] = (wtm_arr[idx] > 0) ? score : -score;\n}\n\n// ── Host API ──────────────────────────────────────────────────────────────\n\nvoid cuda_init() {\n    // Warm up the CUDA runtime (first call to any CUDA function is slow)\n    cudaFree(nullptr);\n}\n\nvoid cuda_cleanup() {\n    cudaDeviceReset();\n}\n\nstd::vector<int> gpu_evaluate_batch(const std::vector<BoardData>& positions) {\n    if (positions.empty()) return {};\n\n    const int n = (int)positions.size();\n\n    // Flatten into contiguous arrays for easy cudaMemcpy\n    std::vector<int8_t> h_boards(n * 64);\n    std::vector<int8_t> h_wtm(n);\n    for (int i = 0; i < n; i++) {\n        std::memcpy(h_boards.data() + i * 64, positions[i].sq, 64);\n        h_wtm[i] = positions[i].wtm;\n    }\n\n    // Allocate device buffers\n    int8_t *d_boards = nullptr, *d_wtm = nullptr;\n    int    *d_scores = nullptr;\n\n    CUDA_CHECK(cudaMalloc(&d_boards, n * 64 * sizeof(int8_t)));\n    CUDA_CHECK(cudaMalloc(&d_wtm,    n      * sizeof(int8_t)));\n    CUDA_CHECK(cudaMalloc(&d_scores, n      * sizeof(int)));\n\n    // Transfer input to GPU\n    CUDA_CHECK(cudaMemcpy(d_boards, h_boards.data(), n * 64, cudaMemcpyHostToDevice));\n    CUDA_CHECK(cudaMemcpy(d_wtm,    h_wtm.data(),    n,      cudaMemcpyHostToDevice));\n\n    // Launch: 256 threads per block, ceil(n/256) blocks\n    constexpr int BLOCK = 256;\n    int grid = (n + BLOCK - 1) / BLOCK;\n    eval_kernel<<<grid, BLOCK>>>(d_boards, d_wtm, d_scores, n);\n    CUDA_CHECK(cudaGetLastError());\n    CUDA_CHECK(cudaDeviceSynchronize());\n\n    // Retrieve results\n    std::vector<int> scores(n);\n    CUDA_CHECK(cudaMemcpy(scores.data(), d_scores, n * sizeof(int), cudaMemcpyDeviceToHost));\n\n    cudaFree(d_boards);\n    cudaFree(d_wtm);\n    cudaFree(d_scores);\n\n    return scores;\n}\n'

files['CMakeLists.txt'] = 'cmake_minimum_required(VERSION 3.18)\nproject(chess LANGUAGES CXX)\n\nset(CMAKE_CXX_STANDARD 17)\nset(CMAKE_CXX_STANDARD_REQUIRED ON)\n\n# ── Build options ─────────────────────────────────────────────────────────\noption(USE_OMP  "Enable OpenMP root-parallel search" ON)\noption(USE_CUDA "Enable CUDA GPU batch evaluation"   OFF)\n\n# ── Common sources ─────────────────────────────────────────────────────────\nset(COMMON_SRC\n    src/board.cpp\n    src/movegen.cpp\n    src/eval.cpp\n    src/search_serial.cpp\n    src/search_omp.cpp\n    src/search_cuda.cpp\n    src/main.cpp\n)\n\n# ── CUDA sources ──────────────────────────────────────────────────────────\nif(USE_CUDA)\n    enable_language(CUDA)\n    set(CMAKE_CUDA_STANDARD 17)\n    set(CMAKE_CUDA_STANDARD_REQUIRED ON)\n    list(APPEND COMMON_SRC cuda/eval_cuda.cu)\nendif()\n\n# ── Executable ────────────────────────────────────────────────────────────\nadd_executable(chess ${COMMON_SRC})\n\ntarget_include_directories(chess PRIVATE src cuda)\n\n# ── Compiler flags ─────────────────────────────────────────────────────────\ntarget_compile_options(chess PRIVATE\n    $<$<COMPILE_LANGUAGE:CXX>:-O3 -march=native -Wall -Wextra -Wno-unused-parameter>\n)\n\nif(USE_CUDA)\n    target_compile_options(chess PRIVATE\n        $<$<COMPILE_LANGUAGE:CUDA>:-O3 --use_fast_math>\n    )\n    set_target_properties(chess PROPERTIES\n        CUDA_SEPARABLE_COMPILATION ON\n    )\n    target_compile_definitions(chess PRIVATE USE_CUDA)\nendif()\n\n# ── OpenMP ────────────────────────────────────────────────────────────────\nif(USE_OMP)\n    # Apple Clang needs libomp from Homebrew; hint CMake to find it\n    if(APPLE)\n        set(OpenMP_CXX_FLAGS "-Xpreprocessor -fopenmp")\n        set(OpenMP_CXX_LIB_NAMES "omp")\n        find_library(OpenMP_omp_LIBRARY omp HINTS /opt/homebrew/opt/libomp/lib)\n        set(OpenMP_CXX_INCLUDE_DIRS "/opt/homebrew/opt/libomp/include")\n    endif()\n    find_package(OpenMP REQUIRED)\n    target_link_libraries(chess PRIVATE OpenMP::OpenMP_CXX)\n    if(APPLE)\n        target_include_directories(chess PRIVATE ${OpenMP_CXX_INCLUDE_DIRS})\n    endif()\n    target_compile_definitions(chess PRIVATE USE_OMP)\nendif()\n\n# ── Install / info ─────────────────────────────────────────────────────────\nmessage(STATUS "Build options:")\nmessage(STATUS "  OpenMP : ${USE_OMP}")\nmessage(STATUS "  CUDA   : ${USE_CUDA}")\nmessage(STATUS "")\nmessage(STATUS "Run targets:")\nmessage(STATUS "  bench (default)   ./chess bench --depth 6")\nmessage(STATUS "  perft             ./chess perft --depth 5")\nmessage(STATUS "  uci               ./chess uci")\n'

for rel, content in files.items():
    path = project_dir / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    print(f'wrote {path} ({len(content)} bytes)')

print(f'\nWrote {len(files)} files into {project_dir}')

## 3. Build With CMake


In [ ]:
import os
import subprocess

cmake_configure = [
    'cmake',
    '-S', '.',
    '-B', 'build',
    '-DUSE_OMP=ON',
    '-DUSE_CUDA=ON',
    f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}',
]
cmake_build = ['cmake', '--build', 'build', '--verbose', f'-j{min(os.cpu_count() or 2, 2)}']

for cmd in (cmake_configure, cmake_build):
    print('$', ' '.join(cmd))
    completed = subprocess.run(cmd, cwd=project_dir, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed: {" ".join(cmd)}')

print('\nBuild complete.')


## 4. Verify Move Generation

These checks confirm the engine still matches known perft totals before benchmarking.


In [ ]:
import subprocess

checks = [
    (
        'Starting position depth 5',
        ['./chess', 'perft', '--depth', '5', '--startpos'],
        '4865609',
    ),
    (
        'Kiwipete depth 4',
        ['./chess', 'perft', '--depth', '4', '--fen', 'r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1'],
        '4085603',
    ),
]

for label, cmd, expected in checks:
    print(f'=== {label} ===')
    result = subprocess.run(cmd, cwd=project_dir / 'build', capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Perft command failed: {cmd}')
    if f'Total: {expected}' not in result.stdout:
        raise AssertionError(f'Expected total {expected} was not found in output for {label}')


## 5. Run Benchmark Sweep


In [ ]:
BENCH_FEN = 'r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1'

import subprocess
from pathlib import Path

report_cmd = ['./chess', 'report', '--fen', BENCH_FEN, '--maxdepth', '8', '--out', 'perf_report.html']
result = subprocess.run(report_cmd, cwd=project_dir / 'build', capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Benchmark run failed')

benchmark_output = result.stdout
(project_dir / 'benchmark_output.txt').write_text(benchmark_output)
print('\nSaved benchmark output to', project_dir / 'benchmark_output.txt')
print('Saved HTML report to', project_dir / 'build' / 'perf_report.html')


## 6. Parse Report Output


In [ ]:
import math
import re

row_pattern = re.compile(
    r'^\s*(\d+)\s+'
    r'([\d.]+)\s+(\d+)\s+(\d+)\s+'
    r'([\d.]+)\s+(\d+)\s+(\d+)\s+'
    r'([\d.]+)×'
    r'(?:\s+CUDA\s+([\d.]+)×)?\s*$'
)

rows = []
for line in benchmark_output.splitlines():
    match = row_pattern.match(line)
    if not match:
        continue
    rows.append(
        {
            'depth': int(match.group(1)),
            'serial_ms': float(match.group(2)),
            'serial_nodes': int(match.group(3)),
            'serial_knps': int(match.group(4)),
            'omp_ms': float(match.group(5)),
            'omp_nodes': int(match.group(6)),
            'omp_knps': int(match.group(7)),
            'omp_speedup': float(match.group(8)),
            'cuda_speedup': float(match.group(9)) if match.group(9) else math.nan,
        }
    )

if not rows:
    raise RuntimeError('Could not parse benchmark table from report output.')

has_cuda = any(not math.isnan(row['cuda_speedup']) for row in rows)
print(f'Parsed {len(rows)} benchmark rows. CUDA data present: {has_cuda}')


## 7. Plot Results


In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

depths = [row['depth'] for row in rows]
serial_ms = [row['serial_ms'] for row in rows]
omp_ms = [row['omp_ms'] for row in rows]
serial_knps = [row['serial_knps'] for row in rows]
omp_knps = [row['omp_knps'] for row in rows]
omp_speedup = [row['omp_speedup'] for row in rows]
cuda_speedup = [row['cuda_speedup'] for row in rows]

plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Chess Engine Performance: Serial vs OpenMP vs CUDA', fontsize=18, y=0.98)

serial_color = '#58a6ff'
omp_color = '#3fb950'
cuda_color = '#bc8cff'
x = np.arange(len(depths))
width = 0.35

axes[0, 0].bar(x - width / 2, serial_ms, width, label='Serial', color=serial_color, alpha=0.85)
axes[0, 0].bar(x + width / 2, omp_ms, width, label='OpenMP', color=omp_color, alpha=0.85)
axes[0, 0].set_title('Search Time (ms)')
axes[0, 0].set_xlabel('Depth')
axes[0, 0].set_ylabel('Milliseconds')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(depths)
axes[0, 0].legend()

axes[0, 1].plot(depths, omp_speedup, 'o-', color=omp_color, linewidth=2, label='OpenMP')
if has_cuda:
    cuda_depths = [row['depth'] for row in rows if not math.isnan(row['cuda_speedup'])]
    cuda_values = [row['cuda_speedup'] for row in rows if not math.isnan(row['cuda_speedup'])]
    axes[0, 1].plot(cuda_depths, cuda_values, 's-', color=cuda_color, linewidth=2, label='CUDA')
axes[0, 1].axhline(1.0, color='#8b949e', linestyle='--', alpha=0.6)
axes[0, 1].set_title('Speedup vs Serial')
axes[0, 1].set_xlabel('Depth')
axes[0, 1].set_ylabel('Speedup (x)')
axes[0, 1].legend()

axes[1, 0].bar(x - width / 2, serial_knps, width, label='Serial', color=serial_color, alpha=0.85)
axes[1, 0].bar(x + width / 2, omp_knps, width, label='OpenMP', color=omp_color, alpha=0.85)
axes[1, 0].set_title('Throughput (kn/ms)')
axes[1, 0].set_xlabel('Depth')
axes[1, 0].set_ylabel('kilo-nodes per ms')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(depths)
axes[1, 0].legend()

axes[1, 1].semilogy(depths, serial_ms, 'o-', color=serial_color, linewidth=2, label='Serial')
axes[1, 1].semilogy(depths, omp_ms, 's-', color=omp_color, linewidth=2, label='OpenMP')
axes[1, 1].set_title('Search Time (log scale)')
axes[1, 1].set_xlabel('Depth')
axes[1, 1].set_ylabel('Milliseconds')
axes[1, 1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda value, _: f'{value:,.0f}'))
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_facecolor('#161b22')
    ax.grid(color='#30363d', alpha=0.25)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plot_path = project_dir / 'chess_benchmark.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved plot to', plot_path)


## 8. Summary Table


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df = df.rename(
    columns={
        'depth': 'Depth',
        'serial_ms': 'Serial (ms)',
        'serial_nodes': 'Serial Nodes',
        'serial_knps': 'Serial kn/ms',
        'omp_ms': 'OpenMP (ms)',
        'omp_nodes': 'OpenMP Nodes',
        'omp_knps': 'OpenMP kn/ms',
        'omp_speedup': 'OpenMP Speedup',
        'cuda_speedup': 'CUDA Speedup',
    }
)

if not has_cuda:
    df = df.drop(columns=['CUDA Speedup'])

styled = (
    df.style
      .format(
          {
              'Serial (ms)': '{:.1f}',
              'OpenMP (ms)': '{:.1f}',
              'OpenMP Speedup': '{:.2f}x',
              **({'CUDA Speedup': '{:.2f}x'} if 'CUDA Speedup' in df.columns else {}),
          },
          na_rep='n/a',
      )
      .format('{:,}', subset=['Serial Nodes', 'OpenMP Nodes'])
      .set_properties(**{'text-align': 'right'})
      .set_table_styles(
          [
              {'selector': 'th', 'props': [('background-color', '#21262d'), ('color', '#8b949e')]},
              {'selector': 'td', 'props': [('border-bottom', '1px solid #30363d')]},
          ]
      )
)

display(styled)
csv_path = project_dir / 'benchmark_summary.csv'
df.to_csv(csv_path, index=False)
print('Saved summary CSV to', csv_path)
